# OCR en Colab — carril sin instalación

**Curso Data Science con Python · Universidad del Pacífico · Semana 10**
*Document AI y voz: OCR, extracción estructurada y ASR*

Este notebook corre el **mismo código** que la versión local. Lo único que cambia es la
máquina: acá hay una GPU T4 gratuita.

> Esa frase es, en sí misma, la lección de la clase.

### Antes de empezar

Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.

Si no lo haces, todo funciona igual pero mucho más lento, y el `doctor` te lo va a
decir en la cara.

## Celda 1 — Instalación

Dos cosas que vas a hacer mal si nadie te las dice:

1. **No hay que instalar CUDA.** PaddlePaddle lo gestiona solo; basta el driver NVIDIA,
   que en Colab ya está. Quien abra la página de descarga de NVIDIA acaba de perder
   media hora.
2. **Nunca instalar `paddlepaddle` y `paddlepaddle-gpu` a la vez.** Conviven sin lanzar
   error, y el resultado es que todo corre en CPU 40 veces más lento **sin avisar**.

Esto tarda unos 3 minutos. No mires la barra: lee la celda siguiente mientras baja.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --extra-index-url https://pypi.org/simple
!python -m pip install -q "paddleocr==3.7.0" "pypdfium2==4.30.0" "pypdf==5.1.0" \
  "pyyaml==6.0.2"
print("Instalación terminada.")

## Celda 2 — Traer el repositorio y el documento

El PDF es público: son las bases de la Licitación Pública de Obras N° 05-2026-C-MPI de
la Municipalidad Provincial de Ilo, descargadas de SEACE.

Todo el material está en el repositorio del curso:

**https://github.com/patrickmelorr-lang/OCR-LICITACIONES**

La celda siguiente lo clona y se mete en la carpeta. Son 5 MB: tarda segundos.

In [ ]:
# Clonar el repositorio del curso
!git clone --quiet https://github.com/patrickmelorr-lang/OCR-LICITACIONES.git ocr_clase
os.chdir("ocr_clase")

for carpeta in ("data/processed", "data/outputs", "logs"):
    Path(carpeta).mkdir(parents=True, exist_ok=True)

print("Carpeta de trabajo:", Path.cwd())
print("Archivos:", sorted(p.name for p in Path.cwd().iterdir()))

## Celda 3 — El doctor: ¿la GPU está **realmente** en uso?

PaddlePaddle **no lanza error cuando la GPU falla**: cae a CPU en silencio y sigue
funcionando. Crees que tienes aceleración y no la tienes.

> Analogía: tienes bicicleta y auto. Le dices al chofer "llévame". Si la llave del auto
> no está colgada, no te avisa: te lleva en bicicleta. Llegas tarde y nunca te enteras
> de que existía el auto.

Seis chequeos. Los cinco primeros pueden dar verde y aun así el modelo correr en CPU.
**El sexto es el único que no se puede falsificar:** hace una inferencia de verdad y
mira si la VRAM se mueve.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))

from src.comun import cargar_config
from src import doctor

config = cargar_config()
for c in doctor.diagnosticar(config):
    print(c.linea())

## Celda 4 — Triaje: la regla de oro

> Usar OCR **solo** en las páginas que son imagen. En el resto, extracción directa, que
> es gratis e instantánea.

Sobre este documento el resultado es extremo y limpio: **57 páginas, 0 con texto, 57
imagen**. Cero por ciento aprovechable sin OCR.

**Pregunta:** si de un documento de 500 páginas solo 20 son escaneadas, ¿cuánto te
ahorras haciendo triaje primero?

In [ ]:
from src import triaje

filas = triaje.ejecutar(config)

## Celda 5 — Preparación: aquí se gana la pelea

Este PDF trae tres problemas de golpe:

1. Entre 9 y 63 **imágenes embebidas por hoja**: el escáner guardó mosaicos. Hay que
   rasterizar la página completa, no extraer sus imágenes.
2. **Dos páginas lógicas por hoja**, una sobre la otra. Hay que partir.
3. El contenido está **girado**. Hay que enderezar.

El orden es: rasterizar → partir → enderezar → OCR. Partir después del OCR no sirve: el
modelo ya mezcló las dos páginas.

In [ ]:
from src import preparar

# Hojas 1 y 2: los cuatro campos no caben en una sola.
generadas = preparar.ejecutar(hoja_inicial=1, hoja_final=2, config=config)
for ruta in generadas:
    print(ruta.name)

In [ ]:
from PIL import Image

pagina = Image.open(generadas[0])
print(f"{generadas[0].name}: {pagina.size[0]}x{pagina.size[1]} px")
pagina.resize((pagina.size[0] // 4, pagina.size[1] // 4))

## Celda 6 — El primer OCR

El OCR no devuelve texto. Devuelve **texto + geometría + confianza**, y los tres datos
valen igual:

| Campo | Para qué sirve |
|---|---|
| `texto` | lo obvio |
| `caja` | resaltar en el original lo que la IA afirma |
| `confianza` | decidir qué revisa un humano |

In [ ]:
from src import ocr_run

motor = ocr_run.construir_ocr(config)
lineas = ocr_run.ocr_imagen(generadas[0], config, motor)

print(f"{len(lineas)} líneas\n")
for linea in lineas[:15]:
    print(f"  {linea.confianza:.3f}  {linea.texto}")

## Celda 7 — El momento GPU: medir sin autoengañarse

> Analogía: cronometras cuánto tarda tu mamá en servir la sopa, pero arrancas el
> cronómetro cuando todavía está prendiendo la cocina. El primer plato siempre parece
> lentísimo.

La primera inferencia incluye cargar el modelo a la VRAM e inicializar CUDA. Por eso el
protocolo **descarta páginas de calentamiento**, mide varias, y reporta la **mediana**,
no el promedio: un pico aislado no debe mover la conclusión.

In [ ]:
from src import bench

# En Colab reducimos la matriz para que quepa en el tiempo de clase.
config["benchmark"]["paginas_calentamiento"] = 1
config["benchmark"]["paginas_medidas"] = 4

mediciones = bench.ejecutar(config, rapido=True)

### Lee la tabla en este orden

1. La **VRAM** sube en modo GPU y queda plana en modo CPU.
2. La **mediana de segundos por página** cae de forma drástica.
3. El **texto reconocido es el mismo**. Solo cambia el tiempo.

Referencia del caso real que da origen a esta clase, sobre una GTX 1660 Ti de 6 GB:
~45 s por página en CPU contra ~1 s en GPU. Un rango de 42 páginas pasó de 32 minutos a
cerca de 1 minuto.

**Honestidad obligatoria:** parte de ese 40x es "GPU buena" y parte es "CPU mal
configurada". Un CPU con inferencia optimizada acorta bastante la brecha. Prometer 40x
sin matices es vender humo.

## Celda 8 — Provocar el fallback mudo

Ahora construimos el motor **sin** el parámetro `device`. Todo va a funcionar. Nada va
a avisar. La VRAM no se va a mover. El tiempo se multiplica.

> Esto es lo más caro que te puede pasar en un proyecto real: creer que tienes
> aceleración y no tenerla. Y fíjate que no hay ningún error en pantalla.
> **El silencio es el problema.**

In [ ]:
import time
from paddleocr import PaddleOCR

base = doctor.leer_vram_usada_mib()
print(f"VRAM antes: {base:.0f} MiB")

# Sin device: PaddleOCR decide por su cuenta, y no avisa qué decidió.
motor_mudo = PaddleOCR(lang="es")

t0 = time.perf_counter()
_ = motor_mudo.predict(str(generadas[0]))
print(f"Tiempo: {time.perf_counter() - t0:.2f} s")

pico = doctor.leer_vram_usada_mib()
print(f"VRAM después: {pico:.0f} MiB (delta {pico - base:+.0f} MiB)")
print()
print("Si el delta es pequeño, el modelo está en CPU y nadie te lo dijo.")
print("Por eso ocr_run.py registra en el log el dispositivo REAL que reporta Paddle,")
print("no el que se le pidió. Lo que importa es dónde corrió.")

## Celda 9 — Extracción de los cuatro campos

De texto a datos. Dos vías, y la comparación **es** la lección.

**La regla de verificación**, heredada del proyecto real:

> Todo dato extraído debe existir **literalmente** en el texto del OCR. Si no existe, se
> marca `sin_verificar`.

Un modelo que "corrige" el número de licitación y acierta es tan peligroso como uno que
lo inventa, porque la próxima vez va a corregir y fallar.

### La clave de API no se escribe en el notebook

Nunca. Ni "solo para probar". Una clave escrita en una celda se guarda dentro del
`.ipynb`, y si ese `.ipynb` llega a un repositorio público la clave queda en el
historial de git **para siempre**. Borrarla después no la saca del historial: hay que
revocarla.

En Colab se pide por teclado con `getpass`, que no muestra lo que escribes ni lo deja
en la salida de la celda. En local, el proyecto la lee de un `.env` que `.gitignore`
excluye.

In [ ]:
import getpass, os

# getpass no muestra lo que se escribe ni lo guarda en la salida de la celda.
if not os.environ.get("ANTHROPIC_API_KEY"):
    escrita = getpass.getpass("Clave de Anthropic (Enter para saltar la via B): ")
    if escrita.strip():
        os.environ["ANTHROPIC_API_KEY"] = escrita.strip()

print("Via B disponible" if os.environ.get("ANTHROPIC_API_KEY")
      else "Sin clave: el notebook sigue con la via A, que no necesita API")

In [ ]:
from src import extraer

# Las cuatro mitades: la entidad esta en el encabezado de la hoja 1 y el objeto,
# el numero y el departamento en la caratula interna, que es la hoja 2.
ocr_run.ocr_lote(generadas, config)

texto_ocr = extraer.texto_de_hojas(config)
campos = config["extraccion"]["campos"]

extracciones = [extraer.extraer_por_reglas(texto_ocr, campos)]
por_modelo = extraer.extraer_por_modelo(texto_ocr, campos)
if por_modelo:
    extracciones.append(por_modelo)

extraer.imprimir_comparacion(extracciones, campos)

## Celda 10 — Cierre

### El dato que contradice la intuición

> En el ranking público de parsing de documentos, los tres primeros puestos de 2026 los
> tienen modelos de **menos de 1.3 mil millones** de parámetros. Un modelo de **235 mil
> millones** queda por debajo.

El pequeño especializado le gana al gigante generalista en su propio terreno.

### Y el más útil para licitaciones

> En Wild-OmniDocBench, que usa **fotos** de las mismas páginas, el mejor modelo cae de
> 91.93 a 72.19. **Veinte puntos** de caída por calidad de imagen.

Traducción: invertir en preprocesamiento rinde más que cambiar de modelo.

### Tu entregable

1. El CSV con los cuatro campos.
2. Una línea: ¿en qué modo corriste, y cuántos segundos por página te tomó?

La segunda parte es la que importa: te obliga a haber medido.

### Sigue con

- `notebooks/01_mini_zaragoza.ipynb` — el pipeline completo paso a paso.
- `fallas/README.md` — tres fallas sembradas; diagnostica leyendo el log.